In [2]:
from sklearn.datasets import fetch_openml

In [3]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.metrics import r2_score, mean_squared_error

In [4]:
import numpy as np

In [5]:
import lightgbm as lgb

In [9]:
from sklearn.datasets import fetch_openml

# 코드 1줄로 다운로드 (as_frame=True로 가져오면 handling하기 매우 편함)

ames = fetch_openml(name="house_prices", as_frame=True, parser="auto")

In [10]:
from sklearn.datasets import load_iris


In [13]:
iris = load_iris()

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [26]:
X = iris.data[:, :3]
y = iris.data[:, 3]

In [15]:
X = ames.data.select_dtypes(include=['number']).fillna(0).drop(columns = ["Id"])
y = ames.target

In [42]:
X = X.to_numpy()
y = y.to_numpy()

In [27]:

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

y_train = y_train.reshape(-1, 1) # type: ignore
y_val = y_val.reshape(-1, 1) # type: ignore
y_test = y_test.reshape(-1, 1) # type: ignore

x_scaler = QuantileTransformer(
n_quantiles=500, 
output_distribution='normal', # FT-Transformer 학습에 유리한 정규분포 변환
random_state=42
)
# x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_val_scaled = x_scaler.transform(X_val)
X_test_scaled = x_scaler.transform(X_test)  

y_train_scaled = y_scaler.fit_transform(y_train)
y_val_scaled = y_scaler.transform(y_val)
y_test_scaled = y_scaler.transform(y_test)  

data = {
"X_train":X_train_scaled,
"X_val":X_val_scaled,
"y_train":y_train_scaled.reshape(-1),
"y_val":y_val_scaled.reshape(-1)
}

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2663: UserWarning: n_quantiles (500) is greater than the total number of samples (96). n_quantiles is set to n_samples.
  warnings.warn(


In [29]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=100,      # 트리의 개수 (넉넉하게 설정)
    learning_rate=0.05,     # 학습률
    random_state=42
)

lgb_model.fit(
    data["X_train"], data["y_train"],
    eval_set=[(data["X_val"], data["y_val"])],
    callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)]
)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000233 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 69
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 3
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LGBMRegressor(learning_rate=0.05, random_state=42)

In [30]:
y_pred = lgb_model.predict(data["X_val"])

y_pred_original = y_scaler.inverse_transform(y_pred.reshape(-1,1)).flatten()
y_val_original = y_scaler.inverse_transform(data["y_val"].reshape(-1,1)).flatten()

r2 = r2_score(data["y_val"], y_pred)
rmse = np.sqrt(mean_squared_error(y_val_original, y_pred_original))

print(f" LightGBM Result -> R2: {r2:.4f} | RMSE: {rmse:.4f}")

 LightGBM Result -> R2: 0.9138 | RMSE: 0.2013
